# No-show Prediction

This notebook analyzes healthcare appointment data to predict patient no-shows. The workflow includes:

- Reading and loading the dataset
- Data cleaning and schema definition
- Handling missing values and dropping unnecessary columns
- Identifying and scaling numeric features for modeling

The goal is to prepare the data for building predictive models to identify factors influencing patient attendance.

This is a backup. Prepping for LR evaluation


In [0]:
%pip install -r ../requirements.txt

In [0]:
import pandas as pd
from pyspark.sql.functions import col

# Databricks
folder_name = '/Workspace/Users/asanders4205@gmail.com/no_show_prediction/input-datasets/'

# Local development
# folder_name = 'input-datasets'

dataset_name = "healthcare_noshows.csv"
dataset_path = f"{folder_name}{dataset_name}"


pdf = pd.read_csv(dataset_path)

In [0]:
access_df = spark.createDataFrame(pdf)

### Define structure and read in dataset

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, BooleanType

patient_schema = StructType([
    StructField('PatientId', IntegerType(), True),
    StructField('AppointmentID', IntegerType(), True),
    StructField('Gender', StringType(), True),
    StructField('ScheduledDay', DateType(), True),
    StructField('AppointmentDay', DateType(), True),
    StructField('Age', IntegerType(), True),
    StructField('Neighborhood', StringType(), True), # Renamed from british spelling 'Neighbourhood'
    StructField('Scholarship', BooleanType(), True),
    StructField('Hypertension', BooleanType(), True), # Renamed from `Hipertension`
    StructField('Diabetes', BooleanType(), True),
    StructField('Alcoholism', BooleanType(), True),
    StructField('Handicap', BooleanType(), True),
    StructField('SMS_received', BooleanType(), True),
    StructField('Showed_up', BooleanType(), True),
    StructField('Date.diff', IntegerType(), True) 
])

In [0]:
''' Read in the csv'''
access_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("nullValue", "null") \
    .schema(patient_schema) \
    .load(dataset_path)


access_df = access_df.withColumnRenamed('Date.diff','date_diff')


### Data Preprocessing

In [0]:
# List integer and boolean columns - how to convert boolean and integer to double at once?
integer_cols = [
    c.name for c in access_df.schema.fields
    if isinstance(c.dataType, (IntegerType, BooleanType))
]

for column in integer_cols:
    access_df = access_df.withColumn(column, col(column).cast("double"))

In [0]:
access_df = access_df.withColumn("Showed_up", col("Showed_up").cast("double"))

In [0]:
from pyspark.sql.types import IntegerType, BooleanType
from pyspark.sql.functions import col
# access_df.printSchema()

# cast_int_to_double(access_df)

In [0]:
from pyspark.sql.functions import col, when, sum as spark_sum


''' Prepare data for modelling
    Find columns with empty recoreds
    Remove rows with > 80% empty fields
'''
# Count missing values per column
missing_counts = access_df.agg(*[
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in access_df.columns
]).first().asDict()

# Display missing value counts as a summary DataFrame
missing_df = spark.createDataFrame(
    [(c, int(v)) for c, v in missing_counts.items()],
    ["column", "missing_count"]
)
display(missing_df.orderBy("missing_count", ascending=False))

In [0]:
access_df = access_df.drop('PatientId', 'AppointmentID')
display(access_df)

In [0]:
# [string_col.name for string_col in access_df.schema.fields if string_col.dataType.typeName() == "string"]

#### Scale numeric values

In [0]:
# Base numeric features (cast to double in the cell above, IDs and target excluded)
numerical_cols = ["Age", "Scholarship", "Hypertension", "Diabetes",
                  "Alcoholism", "Handicap", "SMS_received", "date_diff"]

In [0]:
import math
from pyspark.sql.functions import col, sin, cos, month, dayofweek, dayofyear, lit

access_df = (access_df
    .withColumn("Sched_month_sin",     sin(lit(2 * math.pi) * month(col("ScheduledDay"))      / 12))
    .withColumn("Sched_month_cos",     cos(lit(2 * math.pi) * month(col("ScheduledDay"))      / 12))
    .withColumn("Sched_dayofyear_sin", sin(lit(2 * math.pi) * dayofyear(col("ScheduledDay"))  / 365))
    .withColumn("Sched_dayofyear_cos", cos(lit(2 * math.pi) * dayofyear(col("ScheduledDay"))  / 365))
    .withColumn("Appoi_month_sin",     sin(lit(2 * math.pi) * month(col("AppointmentDay"))    / 12))
    .withColumn("Appoi_month_cos",     cos(lit(2 * math.pi) * month(col("AppointmentDay"))    / 12))
    .withColumn("Appoi_dayofyear_sin", sin(lit(2 * math.pi) * dayofyear(col("AppointmentDay"))/ 365))
    .withColumn("Appoi_dayofyear_cos", cos(lit(2 * math.pi) * dayofyear(col("AppointmentDay"))/ 365))
    .drop("ScheduledDay", "AppointmentDay")   # raw date columns not usable by VectorAssembler
)

# Add the cyclical columns to the feature list defined in the cell above
numerical_cols += [
    "Sched_month_sin", "Sched_month_cos", "Sched_dayofyear_sin", "Sched_dayofyear_cos",
    "Appoi_month_sin", "Appoi_month_cos", "Appoi_dayofyear_sin", "Appoi_dayofyear_cos",
]

# access_df.select(*numerical_cols[-8:]).display()

#### Train-test split

In [0]:
train_df, test_df = access_df.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count():,}  Test: {test_df.count():,}")

In [0]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

# Encode only the 'Gender' column
indexer = StringIndexer(inputCol="Gender", outputCol="Gender_index", handleInvalid="skip")
encoder = OneHotEncoder(inputCol="Gender_index", outputCol="Gender_vec")

numerical_cols += ["Gender_vec"]

In [0]:
vector_assembler = VectorAssembler(
    inputCols=numerical_cols,
    outputCol="features",
    handleInvalid="skip",
)

In [0]:
from sklearn.preprocessing import TargetEncoder
from pyspark.sql.functions import col, create_map, lit
from itertools import chain
import pandas as pd

# display(train_df.limit(5))

# 1. Collect training data to pandas — fit encoder on train only
train_pd = train_df.select("Neighborhood", "Showed_up").toPandas()

enc = TargetEncoder(target_type="binary", smooth="auto")
enc.fit(train_pd[["Neighborhood"]], train_pd["Showed_up"])

# 2. Build a lookup map: Neighborhood string → encoded float
categories   = enc.categories_[0]                  # array of Neighborhood names
encoded_vals = enc.transform(pd.DataFrame({"Neighborhood": categories}))

mapping = dict(zip(categories, encoded_vals[:, 0].tolist()))

# 3. Apply the map to both splits as a new Spark column
map_expr = create_map([lit(x) for x in chain.from_iterable(mapping.items())])

train_df = train_df.withColumn("Neighborhood_te", map_expr[col("Neighborhood")])
test_df  = test_df.withColumn("Neighborhood_te", map_expr[col("Neighborhood")])

# 4. Add to your numerical features and drop the raw string column
numerical_cols += ["Neighborhood_te"]
train_df = train_df.drop("Neighborhood")
test_df  = test_df.drop("Neighborhood")

#### Handle class imbalance
Create a weight column, apply heavy weight to minority class and low weight to majority class

In [0]:
# Find count of no-show appointments and count of show appointments. find quotient and make weights.
from pyspark.sql.functions import col

no_show_count = train_df.filter(col("Showed_up") == 0).count()
show_count = train_df.filter(col("Showed_up") == 1).count()
total_count = train_df.count()

NO_SHOW_RATIO = 0.2
SHOWED_UP_RATIO = 0.8

In [0]:
from pyspark.sql.functions import when

train_df = train_df.withColumn(
    "weightCol",
    when(col("Showed_up") == 1, NO_SHOW_RATIO).
    otherwise(SHOWED_UP_RATIO)
)

In [0]:
%skip
train_df.write.format("delta").mode("overwrite").saveAsTable(name = 'train_df_tbl')

In [0]:
from pyspark.ml.classification import LogisticRegression
lr = LogisticRegression(featuresCol="features", labelCol="Showed_up", maxIter=10, weightCol = "weightCol")

In [0]:
# pipeline = Pipeline(stages=[indexer, encoder, vector_assembler])
stages=[indexer, encoder, vector_assembler, lr]
pipeline = Pipeline().setStages(stages)

In [0]:
%skip
help(LogisticRegression)

In [0]:
%skip
# lr_train_data = train_encoded.select(
#     "features",
#     F.col("Showed_up").alias("label"))

In [0]:
%skip
model = lr.fit(lr_train_data)

In [0]:
%skip
# # Drop string columns that were encoded — the encoded _vec columns are already in the feature vector
# train_encoded = train_encoded.drop('Gender', 'Neighborhood')
# test_encoded  = test_encoded.drop('Gender', 'Neighborhood')

In [0]:
%skip
lr_train_data = train_encoded.select(
    "features",
    F.col("Showed_up").alias("label"))

In [0]:
train_df = spark.read.table('train_df_tbl')

In [0]:
model = pipeline.fit(train_df)

predictions = model.transform(test_df)

In [0]:
%skip
display(train_encoded.select("features"))
display(test_encoded.select("features"))

### Logistic Regression Baseline

No hyperparameter tuning - fast, interpretable baseline to anchor subsequent model comparison

In [0]:
%skip
lr_train_data = train_encoded.select(
    "features",
    F.col("Showed_up").alias("label"))

# display(lr_train_data.limit(5))

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
model = lr.fit(lr_train_data)

In [0]:
%skip
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
model = lr.fit(lr_train_data)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F

In [0]:
test_encoded = model.transform(test_df)

lr_test_data = test_encoded.select(
    "features",
    F.col("Showed_up").alias("label"))

### Need to fit on the test data


In [0]:
%skip
model = lr.fit(lr_test_data)

### Log LR in MLFlow run
SMART Goal: Implement Random Forest classifier as a logged MLflow run. By end of session: model trained, evaluated on AUC/F1, and results visible in MLflow UI alongside the LR baseline.

In [0]:
%skip
# spark.sql("""
# CREATE VOLUME IF NOT EXISTS workspace.default.no_show_volume
# """)

In [0]:
import mlflow       # Experiment tracking & Record ML runs
import mlflow.spark # Log model artifacts
import os
from mlflow.models import infer_signature

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/no_show_volume"

# EXPERIMENT_NAME = "/Users/asanders4205@gmail.com/predict_no_show"
# TARGET = "Showed_up"

### Evaluation of LR

In [0]:
with mlflow.start_run(run_name="LogisticRegressionModel") as run:
    # mlflow.spark.log_model(model, "model") # skipped — Serverless 256MB limit
    mlflow.log_param("maxIter", 10)
    mlflow.log_param("featuresCol", "features")
    mlflow.log_param("labelCol", "label")
    mlflow.log_metric("auc", auc_score)
    mlflow.log_metric("recall", recall)

### Implement Random Forest

In [0]:
%skip
from pyspark.mllib.tree import RandomForest